In [ ]:
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"

!uv pip install -qqq \
    "torch>=2.6.0" "triton>=3.0.0" {install_numpy} pandas \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes "transformers" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels

from huggingface_hub import notebook_login
notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.9 MB/s eta 0:00:00


In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Gemma2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.17 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


In [ ]:
from datasets import load_dataset
import pandas as pd
import re

full_dataset = load_dataset("b-mc2/sql-create-context", split="train")
full_dataset = full_dataset.shuffle(seed=42).select(range(5000))

dataset = full_dataset.train_test_split(test_size=500)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

system_prompt = """You are a problem solving model working on task_description XML block.

<task_description>
You are given a database schema and a natural language question.
Generate the SQL query that answers the question.

CRITICAL RULES:
1. Output ONLY the raw SQL query. No markdown formatting (```sql), no explanations, no chat.
2. DO NOT JOIN tables unless the question explicitly requires data from multiple tables. If the required columns are in ONE table, query ONLY that table.
3. When using GROUP BY, you MUST include the grouping column in the SELECT list.
   - Incorrect: SELECT AVG(salary) FROM employees GROUP BY department
   - Correct: SELECT department, AVG(salary) FROM employees GROUP BY department
4. Use uppercase for SQL formatting (SELECT, FROM, WHERE, GROUP BY, ORDER BY).

EXAMPLES:
Schema: CREATE TABLE products (category TEXT, price REAL);
Question: What is the average price by category?
Correct SQL: SELECT category, AVG(price) FROM products GROUP BY category

Schema: CREATE TABLE users (id INTEGER, city TEXT);
Question: How many users are in each city?
Correct SQL: SELECT city, COUNT(*) FROM users GROUP BY city
</task_description>

You will be given a single task in the question XML block.
Solve only the task in question block.
Generate only the solution, do not generate anything else."""

def formatting_prompts_func(examples):
    questions, contexts, answers = examples["question"], examples["context"], examples["answer"]
    texts = []

    for q, c, a in zip(questions, contexts, answers):
        match = re.search(r"CREATE TABLE (\w+)\s*\(", c, re.IGNORECASE)
        table_name = match.group(1) if match else "unknown_table"

        augmented_schema = f"{c}\n-- Sample: INSERT INTO {table_name} VALUES ('...fake_data_1...');\n-- Sample: INSERT INTO {table_name} VALUES ('...fake_data_2...');"

        combined_question = f"Schema:\n{augmented_schema}\n\nQuestion: {q}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"\nNow for the real task, solve the task in question block.\nGenerate only the solution, do not generate anything else\n<question>{combined_question}</question>\n"},
            {"role": "assistant", "content": a}
        ]

        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:

             messages = [
                {"role": "user", "content": f"{system_prompt}\n\nNow for the real task, solve the task in question block.\nGenerate only the solution, do not generate anything else\n<question>{combined_question}</question>\n"},
                {"role": "assistant", "content": a}
             ]
             text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        texts.append(text)

    return { "text" : texts }

# Map song song
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

model.train()

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        eval_strategy = "steps",
        eval_steps = 150,
        save_strategy = "steps",
        save_steps = 150,
        load_best_model_at_end = True,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        output_dir = "outputs",
        report_to = "none",
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Bắt đầu train V2...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Bắt đầu train V2...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,500 | Num Epochs = 1 | Total steps = 282
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the 

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
150,0.175199,0.169265


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=282, training_loss=0.33243911507281854, metrics={'train_runtime': 2361.0637, 'train_samples_per_second': 1.906, 'train_steps_per_second': 0.119, 'total_flos': 2.609436279538483e+16, 'train_loss': 0.33243911507281854, 'epoch': 1.0})

In [ ]:
hf_model_name = "adamwhite625/gemma-2-2b-text2sql-v2-gguf"

print("Đang nén và đẩy siêu Modle V2 (GGUF)...")
model.push_to_hub_gguf(
    hf_model_name,
    tokenizer,
    quantization_method = "q4_k_m",
    token = True
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Đang nén và đẩy siêu Modle V2 (GGUF)...
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gemma-2-2b-it.Q4_K_M.gguf:   1%|          | 16.0MB / 1.71GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/adamwhite625/gemma-2-2b-text2sql-v2-gguf
Unsloth: Cleaning up temporary files...


'adamwhite625/gemma-2-2b-text2sql-v2-gguf'